# TP4: Spektrogramm und Sprachaktivität
### Labor Versuch TP4: Frequenzen über der Zeit und die Frage "Wann wird gesprochen?"

In diesem Versuch betrachten Sie Sprache, Musik und ein Geräusch als **Spektrogramm** und bauen eine einfache **Sprachaktivitätserkennung**. Sie untersuchen auch, wann diese einfache Methode versagt.

## 1 Hinweise zur Durchführung

- **Begleitheft:** Zu diesem Notebook gehört das PDF [`TP4_Spektrogramm_Sprachaktivitaet.pdf`](https://github.com/Lamhour-Mohamed-Akram/audio-dsp-lab/blob/main/handouts/TP4_Spektrogramm_Sprachaktivitaet.pdf). Abschnitte und Aufgaben haben im Notebook und im PDF **dieselben Nummern**.
- **Pflichtaufgaben** bearbeiten alle. **Zusatzaufgaben** sind freiwillig.
- **Antworten:** Schreiben Sie Ihre Antworten in das gedruckte Begleitheft **oder** in eine eigene Kopie dieses Notebooks. In Google Colab geht das über **Datei > Kopie in Drive speichern** (englisch *File > Save a copy in Drive*). Schreibrechte für das Original brauchen Sie nicht.
- **Zeitbedarf (Schätzung):** Pflichtaufgaben 4.1 bis 4.4 ca. 60 bis 90 Minuten nach der Einrichtung.
- **Arbeitsweise:** Führen Sie die Zellen von oben nach unten mit **Shift + Enter** aus. Ändern Sie in den Aufgaben immer nur **ein oder zwei Parameter** und führen Sie die Zelle danach erneut aus.
- **Kopfhörer:** Tiefe Töne sind über Laptop-Lautsprecher oft kaum hörbar.

**Hinweise für Google Colab**
- Eventuell müssen Sie sich mit einem Google-Konto anmelden.
- Beim ersten Ausführen kann eine Warnung erscheinen, dass das Notebook nicht von Google stammt. Bestätigen Sie mit **"Trotzdem ausführen"** (englisch *Run anyway*).
- **Sitzung neu starten** (englisch *Restart session*): Alle Variablen sind weg, die Dateien bleiben erhalten.
- **Laufzeit trennen und löschen** oder eine **neue Laufzeit**: Auch die heruntergeladenen Dateien sind weg.
- In beiden Fällen: Führen Sie die Zellen in Abschnitt 1 erneut aus und danach alle Zellen bis zu der Stelle, an der Sie arbeiten. Am einfachsten geht das mit **Laufzeit > Alle ausführen** (englisch *Runtime > Run all*).
- Eine GPU wird nicht benötigt. Dieses Notebook braucht kein anderes Notebook.

### 1.1 Einrichtung

Die nächsten zwei Zellen bereiten alles vor. Sie müssen den Code hier noch nicht verstehen.

1. Die erste Zelle prüft, ob alle benötigten Python-Pakete vorhanden sind. Nur fehlende Pakete werden installiert.
2. Die zweite Zelle stellt die Audiodateien `speech.wav`, `music.wav`, `sound.wav` im Ordner `data/` bereit. Sie lädt eine Datei **nur dann** herunter, wenn sie fehlt oder beschädigt ist.

In [ ]:
# Einrichtung 1: fehlende Pakete installieren (in Google Colab ist meist schon alles da)
import importlib.util
import subprocess
import sys

pakete = {"numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib",
          "soundfile": "soundfile", "librosa": "librosa", "IPython": "ipython"}

fehlend = [paket for modul, paket in pakete.items() if importlib.util.find_spec(modul) is None]
if fehlend:
    print("Installiere:", ", ".join(fehlend))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *fehlend])
    print("Fertig.")
else:
    print("Alle Pakete sind bereits installiert.")

In [ ]:
# Einrichtung 2: Audiodateien bereitstellen (Download nur, wenn eine Datei fehlt oder beschädigt ist)
import os
import urllib.request
import soundfile as sf

DATEN_URL = "https://raw.githubusercontent.com/Lamhour-Mohamed-Akram/audio-dsp-lab/afe9e3d4fc8266c132421123ecd92235bc046779/data/"
ERWARTET = {"speech.wav": (16000, 1), "music.wav": (44100, 2), "sound.wav": (22050, 2)}   # Datei: (Abtastrate, Kanäle)


def audiodatei_ok(pfad, abtastrate, kanaele):
    """Prüft, ob die Datei eine lesbare Audiodatei im erwarteten Format ist."""
    try:
        info = sf.info(pfad)
    except Exception:
        return False
    return info.samplerate == abtastrate and info.channels == kanaele and info.frames > 0


os.makedirs("data", exist_ok=True)
for name, (abtastrate, kanaele) in ERWARTET.items():
    pfad = os.path.join("data", name)
    if audiodatei_ok(pfad, abtastrate, kanaele):
        print(name, ": vorhanden")
        continue
    print(name, ": wird heruntergeladen ...")
    try:
        with urllib.request.urlopen(DATEN_URL + name, timeout=60) as antwort:
            inhalt = antwort.read()
    except Exception as fehler:
        raise RuntimeError("Download von " + name + " fehlgeschlagen. Bitte Internetverbindung prüfen "
                           "und diese Zelle erneut ausführen. Grund: " + str(fehler))
    with open(pfad + ".tmp", "wb") as datei:
        datei.write(inhalt)
    if not audiodatei_ok(pfad + ".tmp", abtastrate, kanaele):
        os.remove(pfad + ".tmp")
        raise RuntimeError(name + " wurde geladen, ist aber keine gültige Audiodatei im erwarteten Format.")
    os.replace(pfad + ".tmp", pfad)
    print(name, ": OK")

print("Alle Audiodateien sind bereit.")

### 1.2 Bibliotheken und Hilfsfunktionen

Diese Zelle lädt die Bibliotheken, die drei Aufnahmen und kleine Hilfsfunktionen, die Sie aus TP2 kennen:
- `anhoeren(signal, sr, text, skala)`: Abspieler **ohne** automatische Lautstärke-Anpassung
- `gemeinsame_skala(...)`: ein gemeinsamer Faktor, der Übersteuerung bei der Wiedergabe verhindert
- `sekunden_zu_samples(sekunden, sr)`: Zeit in Samples umrechnen
- `spektrum(signal, sr, glaettung_hz)`: einseitiges Amplitudenspektrum in dB

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import librosa
import librosa.display

from IPython.display import Audio, display

def anhoeren(signal, sr, text="", skala=1.0):
    """Zeigt einen Audio-Abspieler, ohne automatische Lautstärke-Anpassung."""
    signal = np.asarray(signal) * skala
    if np.max(np.abs(signal)) > 1:
        print("Achtung: Werte über 1 werden für die Wiedergabe abgeschnitten. Bitte kleinere Skala wählen.")
        signal = np.clip(signal, -1, 1)
    if signal.ndim == 2:        # Stereo: der Abspieler erwartet (Kanäle, Samples)
        signal = signal.T
    if text:
        print(text)
    display(Audio(signal, rate=sr, normalize=False))

def gemeinsame_skala(*signale):
    """Ein gemeinsamer Faktor für alle Signale. Er ist nur kleiner als 1, wenn sonst etwas übersteuern würde."""
    spitze = max(np.max(np.abs(s)) for s in signale)
    if spitze == 0:
        return 1.0
    return min(1.0, 0.99 / spitze)

def sekunden_zu_samples(sekunden, abtastrate):
    """Rechnet eine Zeit in Sekunden in eine Anzahl von Samples um."""
    anzahl = int(round(sekunden * abtastrate))    # round() rundet, int() macht eine ganze Zahl daraus
    return anzahl

def spektrum(signal, sr, glaettung_hz=0):
    """Einseitiges Amplitudenspektrum in dB. Ein Sinus mit Amplitude 1 ergibt 0 dB."""
    if signal.ndim == 2:
        signal = signal.mean(axis=1)                 # Stereo zu Mono (nur für die Analyse)
    n = len(signal)
    if n < 2:
        raise ValueError("Das Signal ist zu kurz für eine FFT (mindestens 2 Samples).")
    betrag = np.abs(np.fft.rfft(signal)) / n
    betrag[1:] = 2 * betrag[1:]                      # positive Frequenzen: Anteil der negativen dazuzählen
    if n % 2 == 0:
        betrag[-1] = betrag[-1] / 2                  # die Nyquist-Frequenz gibt es nur einmal
    frequenzen = np.fft.rfftfreq(n, d=1 / sr)
    breite = int(glaettung_hz * n / sr)              # Glättung über glaettung_hz Hertz (0 = aus)
    if breite > 1:
        betrag = np.convolve(betrag, np.ones(breite) / breite, mode="same")
    return frequenzen, 20 * np.log10(betrag + 1e-12)

sprache, sr_sprache = sf.read("data/speech.wav")
musik, sr_musik = sf.read("data/music.wav")
geraeusch, sr_geraeusch = sf.read("data/sound.wav")
print("Sprache:", sprache.shape, sr_sprache, "Hz | Musik:", musik.shape, sr_musik, "Hz | Rennwagen:", geraeusch.shape, sr_geraeusch, "Hz")

## 2 Lernziele

Nach diesem Versuch können Sie ...
- den Unterschied zwischen dem **Spektrum der ganzen Aufnahme** und einem **Spektrogramm** erklären,
- Spektrogramme von Sprache, Musik und Geräusch lesen und vergleichen,
- ein **Mel-Spektrogramm** in einfachen Worten erklären,
- mit der **Kurzzeit-Energie** ungefähr erkennen, **wann** gesprochen wird,
- zeigen, wann diese Methode **versagt** (Rauschen, leise Sprache),
- **Sprachaktivitätserkennung**, **Sprechererkennung** und **Spracherkennung** unterscheiden.

## 3 Vorbereitungsaufgaben

- **V1:** Ein Spektrogramm benutzt Fenster von 32 ms. Wie viele Samples sind das bei 16000 Hz?
- **V2:** Das Spektrum der ganzen Aufnahme zeigt nicht, **wann** eine Frequenz vorkommt. Warum ist das bei Sprache ein Nachteil?
- **V3:** Menschen hören den Unterschied zwischen 100 Hz und 200 Hz sehr deutlich, zwischen 8000 Hz und 8100 Hz kaum. Was bedeutet das für eine gehörgerechte Frequenzachse?
- **V4:** Erklären Sie den Unterschied zwischen den Fragen: **Wann** spricht jemand? **Wer** spricht? **Was** wird gesagt?

**Ihre Antworten:**
- V1: 
- V2: 
- V3: 
- V4: 

## 4 Durchführung

### 4.1 Vom Spektrum zum Spektrogramm (Aufgabe 4.1, Pflicht)

Die FFT der **ganzen** Aufnahme mischt alle Zeitpunkte zusammen. Ein **Spektrogramm** geht so vor:
1. Das Signal wird in kurze, überlappende **Fenster** zerlegt.
2. Für jedes Fenster wird eine FFT berechnet.
3. Die Spektren werden nebeneinander als Bild gezeigt.

So lesen Sie das Bild: **x-Achse = Zeit**, **y-Achse = Frequenz**, **Farbe = Stärke** (hell = stark, dunkel = schwach, 0 dB = stärkster Punkt des Bildes).

Parameter: `n_fft` ist die Fensterlänge in Samples (gültig 64 bis 8192), `hop` der Schritt von Fenster zu Fenster (gültig 1 bis `n_fft`).

In [ ]:
def spektrogramm_zeigen(x, sr, titel, n_fft=1024, hop=256, ax=None):
    """Zeigt ein Spektrogramm. Stereo wird nur für die Darstellung zu Mono gemittelt."""
    if not 64 <= n_fft <= 8192:
        raise ValueError("n_fft muss zwischen 64 und 8192 liegen.")
    if not 1 <= hop <= n_fft:
        raise ValueError("hop muss zwischen 1 und n_fft liegen.")
    if x.ndim == 2:
        x = x.mean(axis=1)
    stft = librosa.stft(x, n_fft=n_fft, hop_length=hop)               # Kurzzeit-FFT
    stft_db = librosa.amplitude_to_db(np.abs(stft), ref=np.max)       # in dB, 0 dB = stärkster Punkt
    neue_figur = ax is None
    if neue_figur:
        plt.figure(figsize=(10, 3))
        ax = plt.gca()
    bild = librosa.display.specshow(stft_db, sr=sr, hop_length=hop, n_fft=n_fft, x_axis="time",
                                    y_axis="hz", cmap="magma", vmin=-80, vmax=0, ax=ax)
    plt.colorbar(bild, ax=ax, format="%+2.0f dB")
    ax.set_xlabel("Zeit [s]")
    ax.set_ylabel("Frequenz [Hz]")
    ax.set_title(titel)
    if neue_figur:
        plt.show()

In [ ]:
n_fft = 512      # Fensterlänge in Samples (512 bei 16000 Hz = 32 ms), Vorschläge: 128, 512, 2048
hop = 128        # Schritt in Samples

f1, db1 = spektrum(sprache, sr_sprache, glaettung_hz=20)
plt.figure(figsize=(10, 3))
plt.plot(f1, db1 - np.max(db1), linewidth=0.8)
plt.xlim(0, 8000)
plt.ylim(-80, 5)
plt.xlabel("Frequenz [Hz]")
plt.ylabel("Stärke relativ zum Maximum [dB]")
plt.title("Ganze Aufnahme: geglättetes FFT-Spektrum (keine Zeitinformation)")
plt.grid(True)
plt.show()

spektrogramm_zeigen(sprache, sr_sprache, "Spektrogramm der Sprache", n_fft, hop)

> **Aufgabe 4.1:**
> a) Finden Sie im Spektrogramm die **Pausen** (dunkle senkrechte Streifen). Nennen Sie zwei Zeitpunkte.
> b) In Vokalen sieht man viele **waagerechte Linien** übereinander. Was stellen sie dar?
> c) Setzen Sie `n_fft = 128` und danach `n_fft = 2048`. Was wird jeweils schärfer: die zeitliche Darstellung oder die Frequenzdarstellung?

**Ihre Antworten:**
- a) 
- b) 
- c) 

### 4.2 Sprache, Musik, Geräusch und Mel-Spektrogramm (Aufgabe 4.2, Pflicht)

In [ ]:
fig, achsen = plt.subplots(3, 1, figsize=(10, 9))
spektrogramm_zeigen(sprache, sr_sprache, "Sprache", ax=achsen[0])
spektrogramm_zeigen(musik, sr_musik, "Musik (für die Darstellung zu Mono gemittelt)", ax=achsen[1])
spektrogramm_zeigen(geraeusch, sr_geraeusch, "Rennwagen (für die Darstellung zu Mono gemittelt)", ax=achsen[2])
plt.tight_layout()
plt.show()

**Mel-Spektrogramm.** Das Gehör unterscheidet **tiefe** Frequenzen fein und **hohe** Frequenzen eher grob. Die **Mel-Skala** bildet das nach: Unten sind die Frequenzbänder schmal, oben breit. Ein **Mel-Spektrogramm** fasst die vielen FFT-Frequenzen zu wenigen **Mel-Bändern** zusammen. Das Bild wird kleiner und behält das, was für das Hören wichtig ist. Mel-Spektrogramme werden häufig als **Eingabe für KI-Modelle** benutzt, zum Beispiel für neuronale Netze, die Audio wie ein Bild verarbeiten.

Parameter: `n_mels` ist die Anzahl der Mel-Bänder (gültig 8 bis 128).

In [ ]:
n_mels = 64      # Anzahl der Mel-Bänder, Vorschläge: 20, 64, 128
if not 8 <= n_mels <= 128:
    raise ValueError("n_mels muss zwischen 8 und 128 liegen.")

mel = librosa.feature.melspectrogram(y=sprache, sr=sr_sprache, n_fft=1024, hop_length=160, n_mels=n_mels)
mel_db = librosa.power_to_db(mel, ref=np.max)

plt.figure(figsize=(10, 3))
bild = librosa.display.specshow(mel_db, sr=sr_sprache, hop_length=160, x_axis="time", y_axis="mel",
                                cmap="magma", vmin=-80, vmax=0)
plt.colorbar(bild, format="%+2.0f dB")
plt.xlabel("Zeit [s]")
plt.ylabel("Frequenz [Hz] (Mel-Skala)")
plt.title("Mel-Spektrogramm der Sprache")
plt.show()

print("Spektrogramm mit n_fft = 1024:", 1024 // 2 + 1, "Frequenzen pro Zeitfenster")    # // = ganzzahlig teilen
print("Mel-Spektrogramm:             ", mel_db.shape[0], "Mel-Bänder pro Zeitfenster")

> **Aufgabe 4.2:**
> a) Beschreiben Sie in **Tabelle A** das typische Muster jeder Aufnahme. Achten Sie auf Pausen, lange waagerechte Linien (gehaltene Töne), senkrechte Striche (Schläge, Knacken), steigende oder fallende Linien (Tonhöhe ändert sich) und breites Rauschen.
> b) Setzen Sie `n_mels = 20` und danach `n_mels = 128`. Wie verändert sich das Mel-Spektrogramm? Sind Pausen und Vokale noch zu erkennen?

**Ihre Antwort zu b):** 

### 4.3 Sprachaktivität mit der Kurzzeit-Energie (Aufgabe 4.3, Pflicht)

Wir wollen **ungefähr** herausfinden, **wann** in `speech.wav` gesprochen wird. Diese Aufgabe heißt **Sprachaktivitätserkennung** (englisch *Voice Activity Detection*, VAD).

So funktioniert die einfache Methode:
1. Das Signal wird in kurze, überlappende Fenster zerlegt (25 ms lang, Schritt 10 ms). Das letzte Fenster endet genau am Ende der Aufnahme.
2. Für jedes Fenster wird die **Energie** berechnet: der Mittelwert der quadrierten Samples.
3. Die Energie wird in dB umgerechnet. 0 dB ist das lauteste Fenster. Fenster ohne jede Energie (digitale Stille) bekommen minus unendlich dB.
4. Fenster **über** der Schwelle `schwelle_db` gelten als **aktiv**.
5. Aktive Fenster, die sich zeitlich überlappen oder berühren, werden zu einem **Abschnitt** zusammengefasst. Ein Abschnitt beginnt am Anfang seines ersten aktiven Fensters und endet am Ende seines letzten aktiven Fensters.

Die gefundenen Grenzen sind **ungefähr**. Sie entsprechen **nicht** genau den Wortgrenzen.

In [ ]:
def kurzzeit_energie_db(x, sr, fenster_ms=25, schritt_ms=10):
    """Energie pro Fenster in dB. Gibt Fensteranfänge, Fensterenden (in s) und Energie in dB zurück."""
    if len(x) == 0:
        raise ValueError("Das Signal ist leer.")
    fenster = max(1, min(sekunden_zu_samples(fenster_ms / 1000, sr), len(x)))
    schritt = max(1, sekunden_zu_samples(schritt_ms / 1000, sr))
    anfaenge = list(range(0, len(x) - fenster + 1, schritt))
    if anfaenge[-1] + fenster < len(x):
        anfaenge.append(len(x) - fenster)         # letztes Fenster endet genau am Ende der Aufnahme
    energie = []
    for start in anfaenge:
        stueck = x[start:start + fenster]
        energie.append(np.mean(stueck**2))
    energie = np.array(energie)
    anfang_s = np.array(anfaenge) / sr
    ende_s = (np.array(anfaenge) + fenster) / sr
    energie_db = np.full(len(energie), -np.inf)   # Stille: minus unendlich dB
    if np.max(energie) > 0:
        positiv = energie > 0
        energie_db[positiv] = 10 * np.log10(energie[positiv] / np.max(energie))
    return anfang_s, ende_s, energie_db

In [ ]:
def aktive_abschnitte(anfang_s, ende_s, aktiv):
    """Fasst aktive Fenster, die sich überlappen oder berühren, zu Abschnitten [Beginn, Ende] zusammen."""
    abschnitte = []
    for i in range(len(aktiv)):
        if not aktiv[i]:
            continue
        if len(abschnitte) > 0 and anfang_s[i] <= abschnitte[-1][1]:
            abschnitte[-1][1] = max(abschnitte[-1][1], ende_s[i])     # Abschnitt verlängern
        else:
            abschnitte.append([anfang_s[i], ende_s[i]])                # neuer Abschnitt
    return abschnitte


def sprachaktivitaet(x, sr, schwelle_db, titel):
    """Erkennt aktive Abschnitte, zeichnet sie und gibt sie als Liste zurück."""
    if not -100 <= schwelle_db <= 0:
        raise ValueError("schwelle_db muss zwischen -100 und 0 dB liegen.")
    anfang_s, ende_s, energie_db = kurzzeit_energie_db(x, sr)
    aktiv = energie_db > schwelle_db
    abschnitte = aktive_abschnitte(anfang_s, ende_s, aktiv)

    fig, achsen = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
    achsen[0].plot(np.arange(len(x)) / sr, x, linewidth=0.5)
    for beginn, ende in abschnitte:
        achsen[0].axvspan(beginn, ende, color="orange", alpha=0.3)
    achsen[0].set_title(titel + " (orange = als aktiv erkannt)")
    mitte_s = (anfang_s + ende_s) / 2
    achsen[1].plot(mitte_s, np.maximum(energie_db, -100))     # -100 dB als Untergrenze für das Diagramm
    achsen[1].axhline(schwelle_db, color="red", linestyle="--", label="Schwelle")
    achsen[1].set_ylim(-100, 5)
    achsen[1].set_xlabel("Zeit [s]")
    achsen[1].set_ylabel("Energie [dB]")
    achsen[1].legend(loc="lower right")
    plt.show()

    print("Anzahl aktiver Abschnitte:", len(abschnitte))
    print("Anteil aktiver Fenster:", round(100 * np.mean(aktiv), 1), "%")
    if len(abschnitte) == 0:
        print("Kein aktiver Abschnitt gefunden. Eventuell ist die Schwelle zu hoch oder das Signal still.")
    return abschnitte

In [ ]:
schwelle_db = -30     # Vorschläge: -10, -20, -30, -40, -50 (gültig: -100 bis 0)

abschnitte = sprachaktivitaet(sprache, sr_sprache, schwelle_db, "Sprache")

for beginn, ende in abschnitte[:8]:          # die ersten 8 Abschnitte
    print("aktiv von", round(beginn, 2), "s bis", round(ende, 2), "s")

In [ ]:
# Nur die als aktiv erkannten Abschnitte hintereinander anhören
teile = []
for beginn, ende in abschnitte:
    teile.append(sprache[sekunden_zu_samples(beginn, sr_sprache):sekunden_zu_samples(ende, sr_sprache)])

if len(teile) > 0:
    nur_aktiv = np.concatenate(teile)            # alle Teile aneinanderhängen
    print("Original:", round(len(sprache) / sr_sprache, 1), "s   nur aktiv:", round(len(nur_aktiv) / sr_sprache, 1), "s")
    anhoeren(nur_aktiv, sr_sprache, "Nur die als aktiv erkannten Abschnitte:")
else:
    print("Nichts zum Anhören: kein aktiver Abschnitt.")

> **Aufgabe 4.3:** Probieren Sie `schwelle_db` = -10, -20, -30, -40 und -50. Führen Sie jeweils die beiden Zellen oben aus. Tragen Sie in **Tabelle B** die Anzahl der Abschnitte, den Anteil aktiver Fenster und Ihre Beobachtung ein: Fehlen Teile von Wörtern? Werden Pausen als aktiv markiert?

### 4.4 Grenzen der Methode (Aufgabe 4.4, Pflicht)

**a) Rauschen kann die Erkennung auslösen.** Dieses Notebook erzeugt die verrauschte Sprache selbst, mit festem Startwert `seed=42`.

In [ ]:
vad_snr_db = 10          # Stärke des Rauschens als SNR in dB, Vorschläge: 20, 10, 0

rng = np.random.default_rng(seed=42)
rausch_staerke = np.sqrt(np.mean(sprache**2) / 10**(vad_snr_db / 10))
verrauscht = sprache + rausch_staerke * rng.normal(0, 1, len(sprache))

schwelle_rauschen_db = -30     # Schwelle für die verrauschte Sprache
abschnitte_rauschen = sprachaktivitaet(verrauscht, sr_sprache, schwelle_rauschen_db, "Verrauschte Sprache")
anhoeren(verrauscht, sr_sprache, "Verrauschte Sprache:", gemeinsame_skala(verrauscht))

**b) Leise Sprache kann übersehen werden.** Die zweite Hälfte der Aufnahme wird etwa 30 dB leiser gemacht. Die Schwelle bezieht sich auf das **lauteste Fenster der ganzen Aufnahme**.

In [ ]:
leise_faktor = 0.03             # 0.03 entspricht etwa -30 dB (gültig: 0 bis 1)
schwelle_leise_db = -30         # Schwelle für diese Aufnahme
if not 0 <= leise_faktor <= 1:
    raise ValueError("leise_faktor muss zwischen 0 und 1 liegen.")

leiser_teil = sprache.copy()
mitte = len(sprache) // 2
leiser_teil[mitte:] = leiser_teil[mitte:] * leise_faktor

abschnitte_leise = sprachaktivitaet(leiser_teil, sr_sprache, schwelle_leise_db, "Zweite Hälfte leise")
anhoeren(leiser_teil, sr_sprache, "Zweite Hälfte leise:")

> **Aufgabe 4.4:**
> a) Beobachten Sie bei `vad_snr_db = 10` und `schwelle_rauschen_db = -30`, welche Bereiche als aktiv markiert werden. Suchen Sie eine Schwelle, bei der die Pausen wieder als inaktiv erkannt werden. Tragen Sie sie in **Tabelle C** ein.
> b) Welche Teile der leisen zweiten Hälfte werden bei `schwelle_leise_db = -30` erkannt? Suchen Sie eine Schwelle, bei der die leise Sprache erkannt wird.
> c) Vergleichen Sie Ihre beiden Schwellen. Gibt es eine gemeinsame Schwelle, die für beide Aufnahmen gut funktioniert? Was folgt daraus für diese einfache Methode?

**Zusammenfassung:** Eine Energie-Schwelle unterscheidet nur zwischen "viel Energie" und "wenig Energie". Laute Geräusche wie Rauschen, Husten oder Musik können als Sprache gelten. Leise Sprache kann übersehen werden. Bessere Verfahren nutzen zusätzlich das Spektrum oder trainierte Modelle.

**Ihre Antworten:**
- a) 
- b) 
- c) 

### 4.5 Wann, wer, was? Drei verschiedene Aufgaben

| Aufgabe | Frage | Beispiel für ein Ergebnis | In diesem Labor |
|---|---|---|---|
| **Sprachaktivitätserkennung** (VAD) | **Wann** wird gesprochen? | "aktiv von 0,4 s bis 2,1 s" | ja, einfach, mit der Kurzzeit-Energie |
| **Sprechererkennung** (englisch *speaker identification*) | **Wer** spricht? | "Person A" | nein |
| **Spracherkennung** (englisch *automatic speech recognition*, ASR) | **Was** wird gesagt? | der gesprochene Text | nein |

- Eine Energie-Schwelle sagt **nichts** darüber, **wer** spricht oder **was** gesagt wird. Sie misst nur, ob ein Abschnitt genug Energie hat.
- `speech.wav` enthält **eine einzige** Sprecherstimme. Mit dieser Aufnahme kann man Sprechererkennung weder üben noch testen. Dafür bräuchte man Aufnahmen **mehrerer** bekannter Personen.
- Sprechererkennung und Spracherkennung arbeiten mit **trainierten Modellen**. Als Eingabe dienen oft Mel-Spektrogramme oder MFCCs (Zusatzaufgabe Z4.2). In diesem Labor wird **kein** Modell trainiert.

### 4.6 Zusatzaufgabe Z4.1: Echo (optional)

Ein **Echo** ist eine verzögerte, leisere Kopie des Signals:

Ausgang = Original + `daempfung` mal (Original, um `verzoegerung_ms` später)

Gültig: `verzoegerung_ms` von 1 bis 2000, `daempfung` von 0 bis 1. Beide Stereokanäle bekommen dasselbe Echo, das Stereobild bleibt erhalten.

In [ ]:
verzoegerung_ms = 300    # Vorschläge: 50, 300, 1000
daempfung = 0.5          # Vorschläge: 0.3, 0.8

if not 1 <= verzoegerung_ms <= 2000:
    raise ValueError("verzoegerung_ms muss zwischen 1 und 2000 liegen.")
if not 0 <= daempfung <= 1:
    raise ValueError("daempfung muss zwischen 0 und 1 liegen.")

musik_kurz = musik[:sekunden_zu_samples(8.0, sr_musik)]           # erste 8 Sekunden, beide Kanäle
verzoegerung = sekunden_zu_samples(verzoegerung_ms / 1000, sr_musik)
mit_echo = np.zeros((len(musik_kurz) + verzoegerung, 2))            # Platz für das Echo am Ende
mit_echo[:len(musik_kurz)] += musik_kurz                            # Original
mit_echo[verzoegerung:] += daempfung * musik_kurz                   # verzögerte, leisere Kopie

skala_echo = gemeinsame_skala(musik_kurz, mit_echo)
print("Gemeinsame Skala:", round(skala_echo, 3))
anhoeren(musik_kurz, sr_musik, "Original:", skala_echo)
anhoeren(mit_echo, sr_musik, "Mit Echo:", skala_echo)

> **Z4.1:** Vergleichen Sie `verzoegerung_ms` = 50, 300 und 1000 sowie `daempfung` = 0.3 und 0.8. Wann hören Sie ein deutliches Echo? Wann klingt es eher wie ein Raum oder eine Klangfärbung?

### 4.7 Zusatzaufgabe Z4.2: MFCC (optional)

**MFCCs** (Mel-Frequency Cepstral Coefficients) fassen jedes Fenster des Mel-Spektrogramms in wenigen Zahlen zusammen, hier 13. Sie beschreiben grob die **Klangfarbe**. MFCCs sind klassische Eingaben für Verfahren des Maschinellen Lernens. Wir berechnen und zeigen sie hier nur.

In [ ]:
n_mfcc = 13      # Anzahl der Koeffizienten (gültig: 2 bis 40)
if not 2 <= n_mfcc <= 40:
    raise ValueError("n_mfcc muss zwischen 2 und 40 liegen.")

mfcc = librosa.feature.mfcc(y=sprache, sr=sr_sprache, n_mfcc=n_mfcc, n_fft=1024, hop_length=160)

plt.figure(figsize=(10, 3))
bild = librosa.display.specshow(mfcc, sr=sr_sprache, hop_length=160, x_axis="time", cmap="viridis")
plt.colorbar(bild)
plt.xlabel("Zeit [s]")
plt.ylabel("MFCC-Nummer")
plt.title("MFCCs der Sprache")
plt.show()

print("Form:", mfcc.shape, "= (Koeffizienten, Anzahl Zeitfenster)")

> **Z4.2:** Vergleichen Sie das MFCC-Bild mit dem Mel-Spektrogramm aus 4.2. Wo erkennen Sie die Pausen?

## 5 Auswertung und Diskussion

**Tabelle A: Muster im Spektrogramm (Aufgabe 4.2 a)**

| Aufnahme | Typisches Muster |
|---|---|
| Sprache | |
| Musik | |
| Rennwagen | |

**Tabelle B: Schwelle der Sprachaktivitätserkennung für `speech.wav` (Aufgabe 4.3)**

| schwelle_db | Anzahl Abschnitte | Anteil aktiver Fenster [%] | Beobachtung (Wortteile fehlen? Pausen aktiv?) |
|---|---|---|---|
| -10 | | | |
| -20 | | | |
| -30 | | | |
| -40 | | | |
| -50 | | | |

**Tabelle C: Grenzen der Methode (Aufgabe 4.4)**

| Aufnahme | Einstellung | geeignete Schwelle [dB] | Beobachtung |
|---|---|---|---|
| verrauschte Sprache | vad_snr_db = 10 | | |
| zweite Hälfte leise | leise_faktor = 0.03 | | |

**Fragen**
1. Was zeigt ein Spektrogramm, was das FFT-Spektrum der ganzen Aufnahme nicht zeigt?
2. Was haben Sie bei `n_fft = 128` und `n_fft = 2048` beobachtet? Erklären Sie Ihre Beobachtung mit Zeit und Frequenz.
3. Warum passt die Mel-Skala besser zum menschlichen Hören als eine lineare Frequenzachse?
4. Welche Schwelle fanden Sie für `speech.wav` am geeignetsten? Wie hat sich dieselbe Schwelle bei der verrauschten und bei der leisen Aufnahme verhalten?
5. Warum kann eine Energie-Schwelle **nicht** erkennen, **wer** spricht oder **was** gesagt wird?
6. Nennen Sie eine Anwendung, in der Sprachaktivitätserkennung nützlich ist.

**Ihre Antworten:**
1. 
2. 
3. 
4. 
5. 
6. 

---
## 6 Quellen und Audiodaten

Die Aufnahmen stammen von Dritten. Für dieses Labor wurden nur Ausschnitte herausgeschnitten und als WAV-Dateien gespeichert (20 ms Ein- und Ausblenden an den Schnittstellen, sonst keine Bearbeitung). Alle Details stehen in [`DATA_SOURCES.md`](https://github.com/Lamhour-Mohamed-Akram/audio-dsp-lab/blob/main/DATA_SOURCES.md).

- **`speech.wav`** (Sprache, 16000 Hz, mono, 15,9 s): LibriVox-Hörbuch "Fabeln" von Magnus Gottfried Lichtwer, Abschnitt "Das Wiesel und die Hühner", gelesen von **marham63**. Lizenz: **Public Domain**. Quelle: <https://archive.org/details/fabeln_1905_librivox>. Ausschnitt 12,8 s bis 28,7 s, umgetastet auf 16000 Hz.
- **`music.wav`** (Musik, 44100 Hz, stereo, 20,0 s): Julius Fučík, "Entrance of the Gladiators" ("Einzug der Gladiatoren"), gespielt von der **United States Marine Band**. Lizenz: **Public Domain** (Werk der US-Regierung). Quelle: <https://commons.wikimedia.org/wiki/File:Julius_Fu%C4%8D%C3%ADk%27s_%22Entrance_of_the_Gladiators%22,_performed_by_the_United_States_Marine_Band.flac>. Ausschnitt 24 s bis 44 s, als 16-Bit-WAV gespeichert.
- **`sound.wav`** (Rennwagen, 22050 Hz, stereo, 8,0 s): "Audi R8 (2000)", aufgenommen von **Edvvc** beim Goodwood Festival of Speed 2009. Lizenz: **CC BY-SA 3.0** (<https://creativecommons.org/licenses/by-sa/3.0/>). Quelle: <https://commons.wikimedia.org/wiki/File:Audi_R8_(2000).ogg>. Ausschnitt 4 s bis 12 s, umgetastet auf 22050 Hz. Der Ausschnitt steht ebenfalls unter CC BY-SA 3.0.